[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jrpmichel/Libro_Regularizacion_La_Salle_Bajio/blob/main/int/u3_antiderivada.ipynb)

# Int. U3 · Antiderivada
**Libro de regularización en matemáticas para ingeniería · ULSA Bajío**

Este notebook acompaña la unidad 3 del bloque de cálculo integral. Tiene una sección por subtema, en el mismo orden que el libro:

1. Integral indefinida y constante de integración
2. Tabla de integrales inmediatas (verificador de antiderivadas)
3. Condiciones iniciales

**Cómo usarlo.** Ejecuta las celdas en orden (Entorno de ejecución → Ejecutar todas). Cada sección termina con una celda **Contrasta**: antes de ejecutarla, resuelve a mano el caso que se indica y escribe tu resultado. La referencia de la calculadora aparece solo después de que escribes el tuyo.

**Etiqueta del repositorio.** `INT-U3` (bloque, unidad). Las celdas de código de cada sección vienen de `int/u3_antiderivada/NN_*.py`; los fragmentos que el libro imprime, de `int/u3_antiderivada/fragmentos/`.

**Convenciones.** Punto decimal. Los valores se *redondean* (no se truncan) a las cifras significativas que elijas. Para escribir funciones usa `x` como variable, `*` para multiplicar, `**` o `^` para potencias, `sqrt(...)`, `exp(...)`, `log(...)` (natural) y `abs(...)`. Los ángulos de `sin`, `cos`, `tan` y `sec` van en **radianes**. Toda integral indefinida se muestra con su constante `+ C`.

## 0. Versiones e imports

In [ ]:
# Versiones mínimas probadas. En Colab ya vienen instaladas; esta celda solo lo asegura.
%pip install -q "numpy>=1.26" "sympy>=1.12" "matplotlib>=3.8" "ipywidgets>=8"

import sys, math, re
from decimal import Decimal, ROUND_HALF_UP
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import sympy as sp
import ipywidgets as widgets
from IPython.display import display
from sympy.parsing.sympy_parser import (parse_expr, standard_transformations,
                                        implicit_multiplication_application, convert_xor, rationalize)

print("Python", sys.version.split()[0], "| numpy", np.__version__, "| sympy", sp.__version__,
      "| matplotlib", matplotlib.__version__, "| ipywidgets", widgets.__version__)

In [ ]:
# Símbolos y utilidades compartidas por todo el notebook
x = sp.symbols("x", real=True)

_FUNCIONES = {"sqrt": sp.sqrt, "abs": sp.Abs, "exp": sp.exp, "log": sp.log, "ln": sp.log,
              "pi": sp.pi, "sin": sp.sin, "sen": sp.sin, "cos": sp.cos, "tan": sp.tan, "sec": sp.sec,
              "E": sp.E, "e": sp.E}
_GLOBAL = {"Integer": sp.Integer, "Float": sp.Float, "Rational": sp.Rational,
           "Symbol": sp.Symbol, "Function": sp.Function}
# rationalize: 0.1 se lee como 1/10, para que los resultados salgan exactos
_TRANSF = standard_transformations + (implicit_multiplication_application, convert_xor, rationalize)
_OTRAS_VARIABLES = ("t", "u", "w", "q", "s")      # se leen como x si son la única variable
_C = sp.Symbol("C")


def parsear(texto, variables=("x",)):
    """Convierte un texto en una expresión de sympy o explica qué salió mal.
    Acepta sen y ln, la letra e como número de Euler, una constante + C (se descarta)
    y otra letra (t, u, w, q o s) en lugar de x si es la única variable."""
    if not isinstance(texto, str) or not texto.strip():
        raise ValueError("Escribe una función, por ejemplo: x**2")
    nombres = set(re.findall(r"[A-Za-z_]\w*", texto))
    otras = [v for v in _OTRAS_VARIABLES if v in nombres]
    if "x" not in nombres and len(otras) == 1:
        texto = re.sub(rf"(?<![A-Za-z_]){otras[0]}(?![A-Za-z_0-9])", "x", texto)
        nombres = set(re.findall(r"[A-Za-z_]\w*", texto))
    local = {**_FUNCIONES, "x": x, "C": _C}
    desconocidos = sorted(nombres - set(local))
    if desconocidos:
        raise ValueError(f"No reconozco {desconocidos}. Usa x (o t) como variable y, si hace falta, "
                         "sqrt, exp, log o ln, abs, sin o sen, cos, tan o pi.")
    try:
        expr = parse_expr(texto, local_dict=local, global_dict=dict(_GLOBAL), transformations=_TRANSF)
    except Exception as err:
        raise ValueError("No pude leer la función. Usa * para multiplicar, ** o ^ para potencias "
                         "y sqrt(...) para raíces.") from err
    if not isinstance(expr, sp.Expr) or expr.atoms(sp.core.function.AppliedUndef):
        raise ValueError("Escribe una expresión sin el signo = y solo con las funciones permitidas.")
    expr = expr.subs(_C, 0)                       # la constante de integración no cambia la derivada
    if expr.free_symbols - {x}:
        raise ValueError("Usa una sola variable.")
    return expr


def lista_numeros(texto, nombre="la lista"):
    """'0, 1.5, 3' -> [0.0, 1.5, 3.0], o un mensaje claro si algo no es número."""
    partes = [p for p in re.split(r"[,\s;]+", str(texto).strip()) if p]
    if not partes:
        raise ValueError(f"Escribe {nombre} como números separados por comas, por ejemplo: 0, 1, 3, 4")
    try:
        return [float(p) for p in partes]
    except ValueError as err:
        raise ValueError(f"En {nombre} hay algo que no es un número: usa punto decimal y comas entre valores.") from err


def cifras(valor, n=4):
    """Redondea (no trunca) a n cifras significativas con la regla escolar (5 sube) y conserva los ceros finales."""
    v = float(valor)
    if math.isnan(v):
        return "no definido"
    if math.isinf(v):
        return "+∞" if v > 0 else "-∞"
    if v == 0:
        return "0"
    d = Decimal(repr(v))
    q = d.quantize(Decimal(1).scaleb(d.adjusted() - n + 1), rounding=ROUND_HALF_UP)
    if q.adjusted() != d.adjusted():          # 9.996 con 3 cifras pasa a 10.0: se reajusta la posición
        q = d.quantize(Decimal(1).scaleb(q.adjusted() - n + 1), rounding=ROUND_HALF_UP)
    if q.adjusted() >= n:                     # enteros grandes: sin notación científica
        return f"{int(q):d}"
    return format(q, "f")


def cerca(p, q, tol=1e-9):
    return abs(float(p) - float(q)) < tol


def _rechaza(fn):
    """True si fn() lanza ValueError (para probar que una calculadora rechaza una entrada)."""
    try:
        fn()
    except ValueError:
        return True
    return False


def valor_real(f, c):
    """f(c) como float, o ValueError si no es un número real finito."""
    v = complex(sp.N(f.subs(x, c)))
    if abs(v.imag) > 1e-12 or not math.isfinite(v.real):
        raise ValueError(f"f no tiene valor real en x = {cifras(float(c), 6)}: revisa el dominio.")
    return v.real


def numerica(f):
    """Versión numpy de f que siempre devuelve un arreglo del tamaño de la entrada."""
    g = sp.lambdify(x, f, "numpy")
    return lambda t: np.asarray(g(np.asarray(t, dtype=float)), dtype=float) * np.ones_like(np.asarray(t, dtype=float))

## 1. Integral indefinida y constante de integración

$F$ es antiderivada de $f$ en un intervalo si $F'(x)=f(x)$. Todas las antiderivadas difieren en una constante:

$$\int f(x)\,dx=F(x)+C,\qquad \int x^n\,dx=\frac{x^{n+1}}{n+1}+C\quad(n\neq-1).$$

La calculadora pide a `sympy` una antiderivada, le pone el valor absoluto al logaritmo cuando hace falta ($\ln|x|$), **siempre** escribe la constante $C$ y comprueba que $F'=f$. El control deslizante mueve $C$: la curva sube o baja, pero su pendiente en cada $x$ no cambia.

In [ ]:
_PERMITIDAS = (sp.exp, sp.log, sp.sin, sp.cos, sp.tan, sp.sec, sp.Abs, sp.atan, sp.asin, sp.acos)
_PUNTOS = [-2.9, -2.1, -1.3, -0.7, 0.7, 1.3, 2.1, 2.9]


def con_valor_absoluto(F):
    """ln(u) -> ln|u| salvo que u sea siempre positivo: la antiderivada de u'/u vale donde u > 0 y donde u < 0."""
    return F.replace(sp.log, lambda a: sp.log(a) if (a.is_positive or a.has(sp.Abs)) else sp.log(sp.Abs(a)))


def sin_abs_en_log(f):
    """ln|u| -> ln u, solo para integrar (la derivada de las dos es u'/u); el valor absoluto se repone al final."""
    return f.replace(lambda e: isinstance(e, sp.log) and e.args[0].has(sp.Abs),
                     lambda e: sp.log(e.args[0].replace(sp.Abs, lambda a: a)))


def misma_funcion(g, h):
    """True si g y h coinciden: primero con sympy; si no decide, en los puntos donde las dos son reales."""
    d = sp.simplify(g - h)
    if d == 0:
        return True
    comparados = 0
    for c in _PUNTOS:
        try:
            if abs(valor_real(g, c) - valor_real(h, c)) > 1e-9:
                return False
            comparados += 1
        except (ValueError, TypeError, ZeroDivisionError):
            continue
    return comparados >= 3


def antiderivada(f):
    """Una antiderivada de f (con C = 0), simplificada y con ln|..| donde corresponde."""
    F = sp.integrate(sin_abs_en_log(f), x)
    if F.has(sp.Integral):
        raise ValueError("sympy no encontró una antiderivada para esta función.")
    raras = {type(a).__name__ for a in F.atoms(sp.Function) if not isinstance(a, _PERMITIDAS)}
    if raras:
        raise ValueError(f"la antiderivada usa funciones especiales ({', '.join(sorted(raras))}): no es elemental. "
                         "En la Unidad 4 verás cómo integrar estos casos con métodos numéricos.")
    F = sp.expand(F) if F.is_polynomial(x) else con_valor_absoluto(sp.trigsimp(sp.simplify(F)))
    if not misma_funcion(derivada_legible(F), sin_abs_en_log(f)):
        raise ValueError("la comprobación F' = f falló; revisa la función.")
    return F


def derivada_legible(F):
    """F' sin el Piecewise que sympy agrega al derivar ln|u| (la derivada de ln|u| es u'/u)."""
    G = F.replace(lambda e: isinstance(e, sp.log) and e.args[0].has(sp.Abs),
                  lambda e: sp.log(e.args[0].replace(sp.Abs, lambda a: a)))
    return sp.simplify(sp.diff(G, x))


def aviso_potencia(f):
    """Mensaje si f tiene un término x**(-1), donde la regla de la potencia no aplica."""
    terminos = sp.Add.make_args(sp.expand(f))
    if any(sp.simplify(t * x).is_number for t in terminos if t.has(x)):
        return "Aviso: la regla de la potencia no aplica a x^(-1) (pediría dividir entre 0); la tabla da ln|x|."
    return ""


def calculadora_indefinida(f_txt, C):
    try:
        f = parsear(f_txt)
        F = antiderivada(f)
    except ValueError as err:
        print("Revisa la entrada:", err); return
    print(f"∫ ({f}) dx = {F} + C")
    print("Comprobación: d/dx [", F, "] =", derivada_legible(F))
    if aviso_potencia(f):
        print(aviso_potencia(f))
    Fn = numerica(F)
    xs = np.linspace(-3, 3, 601)
    with np.errstate(all="ignore"):
        fig, ax = plt.subplots(figsize=(5.5, 3))
        for c in (-2, -1, 0, 1, 2):
            ax.plot(xs, Fn(xs) + c, ":", color="0.6", lw=0.9)
        ax.plot(xs, Fn(xs) + C, color="navy", lw=2, label=f"C = {C}")
    ax.axhline(0, color="black", lw=0.6); ax.axvline(0, color="black", lw=0.6)
    ax.set_ylim(-10, 10); ax.set_xlabel("x"); ax.set_ylabel("F(x) + C"); ax.legend(); plt.show()


widgets.interact(calculadora_indefinida,
    f_txt=widgets.Text(value="2*x", description="f(x) ="),
    C=widgets.FloatSlider(value=0, min=-5, max=5, step=0.5, description="C"));

In [ ]:
# Casos de prueba de la sección 1 (resultado conocido)
PRUEBAS_1 = [
    ("x^2 -> x^3/3", lambda: misma_funcion(antiderivada(x**2), x**3 / 3)),
    ("constante 3 -> 3x", lambda: misma_funcion(antiderivada(3 + 0 * x), 3 * x)),
    ("sqrt(x) -> (2/3) x^(3/2)", lambda: misma_funcion(antiderivada(sp.sqrt(x)), sp.Rational(2, 3) * x**sp.Rational(3, 2))),
    ("1/x -> ln|x| y aviso de la regla de la potencia",
     lambda: antiderivada(1 / x) == sp.log(sp.Abs(x)) and aviso_potencia(1 / x) != ""),
    ("entrada vacía o mal escrita se rechaza", lambda: _rechaza(lambda: parsear("")) and _rechaza(lambda: parsear("x^"))),
]
for nombre, prueba in PRUEBAS_1:
    print("OK  " if prueba() else "FALLA", nombre)

**Contrasta (sección 1).** Calcula a mano $\int\big(4x^3-2x+1\big)dx$ y escribe tu respuesta **sin** la constante (la celda la agrega y compara derivando).

In [ ]:
mi_F = None     # escribe tu antiderivada como texto, por ejemplo: "x**2 + 3*x"

if mi_F is None:     # la referencia se muestra después de tu cálculo
    print("falta tu cálculo: escríbelo arriba y vuelve a ejecutar la celda")
else:
    f = 4 * x**3 - 2 * x + 1
    print("La calculadora da:", antiderivada(f), "+ C")
    try:
        ok = misma_funcion(sp.diff(parsear(str(mi_F)), x), f)
    except ValueError as err:
        print(err); ok = False
    print("coinciden (salvo la constante)" if ok else
          "NO coinciden: integra término a término con la regla de la potencia y deriva tu resultado para comprobar")

## 2. Tabla de integrales inmediatas y verificador de antiderivadas

| $f(x)$ | $\int f(x)\,dx$ |
|---|---|
| $x^n$, $n\neq-1$ | $\frac{x^{n+1}}{n+1}+C$ |
| $\frac1x$ | $\ln\lvert x\rvert+C$ |
| $e^x$ | $e^x+C$ |
| $a^x$ | $\frac{a^x}{\ln a}+C$ |
| $\sen x$ | $-\cos x+C$ |
| $\cos x$ | $\sen x+C$ |
| $\frac{1}{\cos^2x}$ | $\tan x+C$ |

Regla del argumento lineal: $\int f(ax+b)\,dx=\frac1aF(ax+b)+C$.

El **verificador** recibe $f$ y la antiderivada que tú propones. Deriva tu propuesta y la compara con $f$, primero con `sympy` y, si hace falta, en ocho puntos de $[-3,3]$. Te dice si tu respuesta es correcta, si solo vale en parte del dominio de $f$ (por ejemplo, un `log(x)` sin valor absoluto) o si es incorrecta, y en ese caso cuánto vale $F'-f$. Si tu respuesta es correcta pero se ve distinta de la de `sympy`, te dice en qué constante difieren.

In [ ]:
def verificar(f, F):
    """('correcta' | 'parcial' | 'incorrecta', detalle) para la antiderivada propuesta F de f."""
    dF = derivada_legible(F)
    simbolico = sp.simplify(dF - sin_abs_en_log(f)) == 0
    falla_F, compara, distintos = [], 0, []
    for c in _PUNTOS:
        if not _definida(f, c):
            continue                                  # f no existe ahí: no cuenta
        if not _definida(F, c):
            falla_F.append(c); continue               # f existe pero tu F no
        fc, dc = valor_real(f, c), valor_real(dF, c)
        compara += 1
        if abs(dc - fc) > 1e-9 * max(1, abs(fc)):     # tolerancia relativa
            distintos.append(c)
    if simbolico or (compara >= 3 and not distintos):
        if falla_F:
            lista = ", ".join(cifras(c, 2) for c in falla_F)
            return "parcial", (f"F' = f donde tu F existe, pero tu F no existe en x = {lista}, donde f sí: "
                               "falta un valor absoluto o tu F no cubre todo el dominio de f.")
        G = antiderivada(f)
        dif = sp.simplify(F - G)
        extra = f" Difiere de la de sympy, {G}, en la constante {dif}." if dif != 0 and dif.is_number else ""
        return "correcta", "F' coincide con f." + extra
    detalle = f"F' - f = {sp.simplify(dF - f)}"
    if distintos:
        c = distintos[0]
        detalle += f"; en x = {c} vale {cifras(valor_real(dF, c) - valor_real(f, c), 6)}"
    return "incorrecta", detalle


def _definida(expr, c):
    try:
        valor_real(expr, c); return True
    except (ValueError, TypeError, ZeroDivisionError):
        return False


_TEXTO = {"correcta": "CORRECTA", "parcial": "CORRECTA SOLO EN PARTE DEL DOMINIO", "incorrecta": "INCORRECTA"}


def calculadora_verificador(f_txt, F_txt):
    try:
        f, F = parsear(f_txt), parsear(F_txt)
        veredicto, detalle = verificar(f, F)
    except ValueError as err:
        print("Revisa la entrada:", err); return
    print(f"f(x) = {f}\ntu F(x) = {F}\nF'(x) = {derivada_legible(F)}")
    print("Veredicto:", _TEXTO[veredicto], "·", detalle)
    print("Recuerda escribir la respuesta con + C. Ángulos en radianes.")


widgets.interact(calculadora_verificador,
    f_txt=widgets.Text(value="cos(2*x)", description="f(x) ="),
    F_txt=widgets.Text(value="sin(2*x)", description="tu F(x) ="));

In [ ]:
# Casos de prueba de la sección 2 (resultado conocido)
PRUEBAS_2 = [
    ("3x^2 con F = x^3 + 7: correcta", lambda: verificar(3 * x**2, x**3 + 7)[0] == "correcta"),
    ("cos 2x con F = sin 2x: incorrecta", lambda: verificar(sp.cos(2 * x), sp.sin(2 * x))[0] == "incorrecta"),
    ("1/x con F = ln|x| + 1: correcta", lambda: verificar(1 / x, sp.log(sp.Abs(x)) + 1)[0] == "correcta"),
    ("1/x con F = ln x: correcta solo en parte del dominio", lambda: verificar(1 / x, sp.log(x))[0] == "parcial"),
    ("(x+1)^2 con F = (x+1)^3/3: correcta y difiere de sympy en 1/3",
     lambda: verificar((x + 1)**2, (x + 1)**3 / 3)[0] == "correcta" and "1/3" in verificar((x + 1)**2, (x + 1)**3 / 3)[1]),
    ("1/cos^2 x con F = tan x: correcta", lambda: verificar(1 / sp.cos(x)**2, sp.tan(x))[0] == "correcta"),
]
for nombre, prueba in PRUEBAS_2:
    print("OK  " if prueba() else "FALLA", nombre)

**Contrasta (sección 2).** Calcula a mano $\int\big(e^{2x}+\frac1x\big)dx$ y escribe tu antiderivada.

In [ ]:
mi_F2 = None     # escribe tu antiderivada como texto, sin + C

if mi_F2 is None:     # la referencia se muestra después de tu cálculo
    print("falta tu cálculo: escríbelo arriba y vuelve a ejecutar la celda")
else:
    f = sp.exp(2 * x) + 1 / x
    try:
        v, d = verificar(f, parsear(str(mi_F2)))
    except ValueError as err:
        v, d = "incorrecta", str(err)
    print("La calculadora da:", antiderivada(f), "+ C")
    print("Tu respuesta:", _TEXTO[v], "·", d)
    if v != "correcta":
        print("Revisa el 1/2 de la regla del argumento lineal y el valor absoluto del logaritmo.")

## 3. Condiciones iniciales

* Primer orden: $y'=f(x)$, $y(x_0)=y_0$. Solución $y=F(x)+C$ con $C=y_0-F(x_0)$.
* Segundo orden: $y''=g(x)$, $y'(x_0)=v_0$, $y(x_0)=y_0$. Se integra dos veces y cada constante se fija **en cuanto aparece**.

La calculadora muestra cada constante despejada, la solución y su gráfica. Rechaza un $x_0$ donde $f$ no está definida y recuerda que la solución vale en el intervalo que contiene a $x_0$.

In [ ]:
def resolver_pvi(f, x0, y0, orden=1, v0=0):
    """(y, y', constantes) del problema de valor inicial de primer o segundo orden."""
    x0 = sp.nsimplify(x0); y0 = sp.nsimplify(y0); v0 = sp.nsimplify(v0)
    valor_real(f, x0)                                   # ValueError si f no existe en x0
    if orden == 1:
        F = antiderivada(f)
        C = sp.simplify(y0 - F.subs(x, x0))
        return _ordenar(F + C), f, {"C": C}
    G = antiderivada(f)
    C1 = sp.simplify(v0 - G.subs(x, x0))
    yp = G + C1
    Y = antiderivada(yp) if yp.has(x) else yp * x
    C2 = sp.simplify(y0 - Y.subs(x, x0))
    return _ordenar(Y + C2), _ordenar(yp), {"C1": C1, "C2": C2}


def _ordenar(e):
    """Polinomios desarrollados; lo demás, simplificado."""
    return sp.expand(e) if e.is_polynomial(x) else sp.simplify(e)


def calculadora_pvi(f_txt, orden, x0, y0, v0, x_eval, n_cifras):
    try:
        f = parsear(f_txt)
        y, yp, cons = resolver_pvi(f, x0, y0, 1 if orden.startswith("1") else 2, v0)
    except ValueError as err:
        print("Revisa la entrada:", err); return
    for nombre, valor in cons.items():
        print(f"{nombre} = {valor}")
    print("y(x) =", y)
    if not orden.startswith("1"):
        print("y'(x) =", yp)
    try:
        print(f"y({x_eval}) ≈ {cifras(valor_real(y, x_eval), n_cifras)}   ({n_cifras} cifras significativas, redondeado)")
    except ValueError as err:
        print("No se puede evaluar en ese punto:", err)
    print("La solución vale en el intervalo que contiene a x0 donde f es continua.")
    xs = np.linspace(float(x0) - 3, float(x0) + 3, 400)
    with np.errstate(all="ignore"):
        fig, ax = plt.subplots(figsize=(5.5, 3))
        ax.plot(xs, numerica(y)(xs), color="navy", lw=2, label="y(x)")
        if not orden.startswith("1"):
            ax.plot(xs, numerica(yp)(xs), "--", color="red", label="y'(x)")
    ax.plot([float(x0)], [float(y0)], "o", color="red")
    ax.axhline(0, color="black", lw=0.6); ax.set_xlabel("x"); ax.legend(); plt.show()


widgets.interact(calculadora_pvi,
    f_txt=widgets.Text(value="3*x**2 - 3", description="f o g ="),
    orden=widgets.Dropdown(options=["1: y' = f", "2: y'' = g"], value="1: y' = f", description="orden"),
    x0=widgets.FloatText(value=2, description="x0"), y0=widgets.FloatText(value=5, description="y0"),
    v0=widgets.FloatText(value=0, description="y'(x0)"),
    x_eval=widgets.FloatText(value=3, description="evaluar en"),
    n_cifras=widgets.IntSlider(value=5, min=1, max=10, description="cifras sig."));

In [ ]:
# Casos de prueba de la sección 3 (resultado conocido)
PRUEBAS_3 = [
    ("y' = 2x, y(0) = 1 -> x^2 + 1", lambda: misma_funcion(resolver_pvi(2 * x, 0, 1)[0], x**2 + 1)),
    ("y'' = -9.8, y'(0) = 12, y(0) = 1.5 -> y(1) = 8.6",
     lambda: cerca(valor_real(resolver_pvi(sp.Rational(-49, 5) + 0 * x, 0, 1.5, 2, 12)[0], 1), 8.6)),
    ("y' = 1/x, y(1) = 2 -> y(e) = 3", lambda: cerca(valor_real(resolver_pvi(1 / x, 1, 2)[0], math.e), 3)),
    ("y' = 1/x con x0 = 0 se rechaza", lambda: _rechaza(lambda: resolver_pvi(1 / x, 0, 1))),
    ("y' = cos x, y(0) = 0 -> sen x", lambda: misma_funcion(resolver_pvi(sp.cos(x), 0, 0)[0], sp.sin(x))),
    ("y'' = 6x, y'(0) = 0, y(0) = 0 -> x^3", lambda: misma_funcion(resolver_pvi(6 * x, 0, 0, 2, 0)[0], x**3)),
]
for nombre, prueba in PRUEBAS_3:
    print("OK  " if prueba() else "FALLA", nombre)

**Contrasta (sección 3).** Resuelve a mano $y'=3x^2$ con $y(1)=5$.

In [ ]:
mi_y = None     # escribe tu solución como texto, por ejemplo: "x**3 + 1"

if mi_y is None:     # la referencia se muestra después de tu cálculo
    print("falta tu cálculo: escríbelo arriba y vuelve a ejecutar la celda")
else:
    ref = resolver_pvi(3 * x**2, 1, 5)[0]
    print("La calculadora da: y =", ref)
    try:
        ok = misma_funcion(parsear(str(mi_y)), ref)
    except ValueError as err:
        print(err); ok = False
    print("coinciden" if ok else "NO coinciden: después de integrar, sustituye x = 1 y despeja C para que y(1) = 5")

## Verificación final
Ejecuta esta celda al terminar. Debe imprimir que todas las pruebas pasaron.

In [ ]:
todas = PRUEBAS_1 + PRUEBAS_2 + PRUEBAS_3
fallas = [nombre for nombre, prueba in todas if not prueba()]
assert not fallas, f"Fallan {len(fallas)} pruebas: {fallas}"
print(f"Todas las verificaciones pasaron: {len(todas)}/{len(todas)}")